# Hidden-unit stimulus selectivity — ACE model-size sweep

Analyses stimulus tuning of individual hidden units for a chosen variant from
`results/22_04_26_ace_tpp1000_model_sweep`.

**Workflow**
1. Pick a variant (hidden size) and set a trial-selection window using *global* trial indices.
2. For each run, condition-average stim-window hidden states over the selected trials.
3. Compute a selectivity index (SI) per unit across stimuli.
4. Define selective units by an SI threshold and plot tuning heatmaps, SI distributions,
   and selective-unit proportions — per run and averaged.
5. Sweep the SI threshold to see how proportions change.

**Stimulus set (ACE task):** A (H→L, idx 0) · C (50%, idx 2) · E (L→H, idx 4)  
**Phase structure:** 0=pre-train | 1=post-train | 2=pre-plast-OFF | 3=post-plast-OFF

**Trial-selection note:** global trial indices are the same x-axis as the behaviour
plots (0 … total_trials−1). Plasticity switches OFF at trial `plasticity_off_trial_idx`
(printed in the config cell below). Set `SELECT_PHASES` and/or `TRIAL_MIN` / `TRIAL_MAX`
to restrict which trials enter the SI computation.

In [ ]:
import pickle
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from collections import defaultdict

%matplotlib inline

In [ ]:
# ── Stimulus colours ─────────────────────────────────────────────────────────
STIM_COLORS = {
    0: '#e41a1c',   # A (H→L)
    2: '#4daf4a',   # C (50%)
    4: '#984ea3',   # E (L→H)
}
LINESTYLES_STIM = {0: '-', 2: ':', 4: '-'}

CTX_COLORS  = {0: '#2166ac', 1: '#d6604d'}   # blue=pre, red=post
CTX_TITLES  = {0: 'Context 0 (pre-reversal)', 1: 'Context 1 (post-reversal)'}

plt.rcParams.update({
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'legend.frameon':     False,
    'font.size':          11,
    'axes.labelsize':     11,
    'axes.titlesize':     11,
    'legend.fontsize':    9,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
})

PLOT_DIR = None   # set by config cell

def save_fig(fig, name):
    if PLOT_DIR is None:
        return
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    out = PLOT_DIR / name
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')

---
## 0. Config — edit here

**Step 1:** choose a `VARIANT_IDX` from the list printed below.  
**Step 2:** set `SELECT_PHASES` and/or `TRIAL_MIN` / `TRIAL_MAX` to restrict which
trials are used when computing condition-averaged hidden states (and therefore SI).

Useful values (printed by the cell):
- `plasticity_off_trial_idx` — where plasticity switches OFF (start of phase 2)
- reversal trial indices — where reward contingencies flip

Setting `SELECT_PHASES = None` and `TRIAL_MIN = TRIAL_MAX = None` uses **all** trials.

In [ ]:
RESULTS_DIR = Path('../results/22_04_26_ace_tpp1000_model_sweep')

variant_dirs = sorted([d for d in RESULTS_DIR.iterdir() if d.is_dir()])
print('Available variants:')
for i, d in enumerate(variant_dirs):
    print(f'  [{i}] {d.name}')

# ─────────────────────────────────────────────────────────────────────────────
VARIANT_IDX = -1    # index into list above  (-1 = last = largest hidden size)

# Trial selection — which *phases* to include (list of phase_idx 0–3, or None = all)
# Phase 0 = pre-train (plast ON)  |  Phase 1 = post-train (plast ON)
# Phase 2 = pre-plast-OFF         |  Phase 3 = post-plast-OFF
SELECT_PHASES = [2, 3]     # None to include all phases

# Additional global-trial-index range filter (applied on top of SELECT_PHASES).
# Set to None for no extra restriction.
TRIAL_MIN = None   # inclusive lower bound
TRIAL_MAX = None   # inclusive upper bound

# Reversal-phase granularity for condition averaging:
#   'reversal_phase'  (0=pre, 1=post reversal; 2 conditions)
#   'phase_idx'       (0–3; up to 4 conditions)
COND_BY = 'reversal_phase'
# ─────────────────────────────────────────────────────────────────────────────

VARIANT_DIR = variant_dirs[VARIANT_IDX]
PLOT_DIR    = VARIANT_DIR / 'selectivity_plots'

run_dirs = sorted([d for d in VARIANT_DIR.iterdir()
                   if d.is_dir() and d.name.startswith('run_')])
print(f'\nSelected: {VARIANT_DIR.name}')
print(f'Runs:      {len(run_dirs)}')
print(f'Plots →    {PLOT_DIR}')

In [ ]:
cfg = {}
with open(VARIANT_DIR / 'config.csv') as f:
    for row in csv.DictReader(f):
        cfg[row['param']] = row['value']

n_train_phases = int(cfg['n_train_phases'])
n_test_phases  = int(cfg['n_test_phases'])
hidden_size    = int(cfg['hidden_size'])
readout_size   = int(cfg['readout_size'])

print(f'hidden_size={hidden_size}  readout_size={readout_size}')
print(f'n_train_phases={n_train_phases}  n_test_phases={n_test_phases}')

PLOT_STIMS  = [0, 2, 4]                    # ACE: A, C, E
STIM_NAMES  = {0: 'A (H→L)', 2: 'C (50%)', 4: 'E (L→H)'}
STIM_LETTER = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F'}
SMOOTH      = 20

# Sanity check: print phase/trial info from first run
with open(run_dirs[0] / 'lick_value_data.pkl', 'rb') as f:
    _lv0 = pickle.load(f)
print(f'\nFrom run 0:')
print(f'  plasticity_off_trial_idx = {_lv0["plasticity_off_trial_idx"]}')
print(f'  rev_indices_global       = {_lv0["rev_indices_global"]}')
print(f'  SELECT_PHASES            = {SELECT_PHASES}')
print(f'  TRIAL_MIN / TRIAL_MAX    = {TRIAL_MIN} / {TRIAL_MAX}')

# Print approximate trial-index ranges per phase (from first run)
with open(run_dirs[0] / 'metrics_numpy.pkl', 'rb') as f:
    _m0 = pickle.load(f)
from collections import defaultdict
_phase_ranges = defaultdict(list)
for tb in _m0['trial_boundaries']:
    _phase_ranges[tb['phase_idx']].append(tb['trial_idx'])
print('\nApproximate global trial-index ranges per phase (run 0):')
for ph, idxs in sorted(_phase_ranges.items()):
    print(f'  phase {ph}: trials {min(idxs)} – {max(idxs)}  (n={len(idxs)})')
del _lv0, _m0, _phase_ranges

---
## 1. Load behaviour data & quick overview

In [ ]:
all_lv = []
for rd in run_dirs:
    p = rd / 'lick_value_data.pkl'
    if p.exists():
        with open(p, 'rb') as f:
            all_lv.append(pickle.load(f))
    else:
        print(f'WARNING: missing {p}')
print(f'Loaded lick_value_data from {len(all_lv)} / {len(run_dirs)} runs')

# Convenience: median plasticity-OFF onset and reversal trials
plast_off_med = int(np.median([lv['plasticity_off_trial_idx'] for lv in all_lv
                                if lv.get('plasticity_off_trial_idx') is not None]))
rev_med = []
rev_lists = [lv['rev_indices_global'] for lv in all_lv if lv.get('rev_indices_global')]
if rev_lists:
    n_rev = min(len(r) for r in rev_lists)
    rev_med = [int(np.median([r[i] for r in rev_lists])) for i in range(n_rev)]
print(f'Median plasticity-OFF onset trial: {plast_off_med}')
print(f'Median reversal trial indices:     {rev_med}')

In [ ]:
def _add_markers(ax, *, plast_off=plast_off_med, rev_trials=rev_med):
    for ri, rt in enumerate(rev_trials):
        ax.axvline(rt, color='k', ls=':', lw=0.8, alpha=0.7,
                   label='Reversal' if ri == 0 else None)
        ax.text(rt, 1.01, f'R{ri+1}', fontsize=6, ha='center',
                transform=ax.get_xaxis_transform(), color='k')
    ax.axvline(plast_off, color='navy', ls='--', lw=2,
               label='Plasticity OFF', zorder=6)
    ax.text(plast_off, 1.05, 'plast.OFF', fontsize=7, ha='center',
            transform=ax.get_xaxis_transform(), color='navy', fontweight='bold')

# Compute x_max from all loaded runs
x_max = max(
    max(np.array(x).max() for lv in all_lv
        for x in lv.get('global_trial_indices_by_stim', {}).values() if len(x) > 0),
    default=plast_off_med * 2
)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
for ax, data_key, ylabel in zip(
        axes,
        ['trial_lick_probs', 'trial_values'],
        ['Lick probability', 'Value estimate']):
    for stim_idx in PLOT_STIMS:
        k   = STIM_LETTER[stim_idx]
        col = STIM_COLORS[stim_idx]
        ls  = LINESTYLES_STIM[stim_idx]
        run_series = []
        for lv in all_lv:
            y = lv.get(data_key, {}).get(k)
            x = lv.get('global_trial_indices_by_stim', {}).get(k)
            if y is None or len(y) == 0:
                continue
            idx = np.array(x)[:len(y)] if (x is not None and len(x) >= len(y)) else np.arange(len(y))
            sm  = pd.Series(np.array(y, dtype=float)).rolling(SMOOTH, min_periods=1).mean().values
            ax.plot(idx, sm, color=col, lw=0.5, alpha=0.15, ls=ls)
            run_series.append(pd.Series(sm, index=idx))
        if not run_series:
            continue
        all_x  = np.unique(np.concatenate([s.index.values for s in run_series]))
        mean_y = np.nanmean(np.stack([s.reindex(all_x).to_numpy() for s in run_series]), axis=0)
        ax.plot(all_x, mean_y, color=col, lw=2, ls=ls,
                label=STIM_NAMES[stim_idx], zorder=4)
    _add_markers(ax)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, x_max)

# Shade the selected trial window on both axes
_lo = TRIAL_MIN if TRIAL_MIN is not None else 0
_hi = TRIAL_MAX if TRIAL_MAX is not None else x_max
if SELECT_PHASES is not None or TRIAL_MIN is not None or TRIAL_MAX is not None:
    for ax in axes:
        ax.axvspan(_lo, _hi, color='gold', alpha=0.12, zorder=0, label='Selected trials')

axes[1].set_xlabel('Global trial index')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(),
           fontsize=8, ncol=1, loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.suptitle(f'Behaviour overview — {VARIANT_DIR.name}\n'
             f'Gold shading = trial window used for SI analysis  '
             f'(phases={SELECT_PHASES}, [{TRIAL_MIN},{TRIAL_MAX}])',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '0_behaviour_overview.png')
plt.show()

---
## 2. Condition-average hidden states & compute SI

For each run, for each trial in the selected window:
- Extract the mean hidden state over the stim-window timesteps.
- Group by `(stimulus, context)` — context = `reversal_phase` (0/1) or `phase_idx` (0–3)
  depending on `COND_BY`.
- Average within each group → **tuning matrix** `T[stim, unit]`.

$$SI_u = \frac{r_{\max,u} - r_{\min,u}}{|r_{\max,u}| + |r_{\min,u}| + \varepsilon}$$

SI = 1 → unit responds to exactly one stimulus; SI ≈ 0 → flat tuning.  
**Note:** no plasticity-on filter is applied — selection is entirely driven by
`SELECT_PHASES` and `TRIAL_MIN` / `TRIAL_MAX`.

In [ ]:
_STIM_STATES = set('ABCDEF')
SILENT_THR   = 1e-4   # units whose max activation < this are called 'silent'


def compute_run_tuning(run_dir, plot_stims=PLOT_STIMS,
                       select_phases=None, trial_min=None, trial_max=None,
                       cond_by='reversal_phase', balance=True):
    """
    Load metrics_numpy.pkl and compute condition-averaged stim-window hidden states.

    Parameters
    ----------
    select_phases : list[int] or None
        Which phase_idx values to include (0–3).  None = all.
    trial_min / trial_max : int or None
        Inclusive global-trial-index bounds (None = no limit).
    cond_by : 'reversal_phase' | 'phase_idx'
        Key used to define separate conditions for condition-averaging.
    balance : bool
        If True, truncate every (stim, cond) cell to the minimum count across
        all cells before averaging.  Trials are taken in chronological order
        (first N) so the result is deterministic.

    Returns
    -------
    dict with keys:
        cond_means    : dict (stim_idx, cond) -> (H,)  mean hidden state
        n_trials_raw  : dict (stim_idx, cond) -> int   raw count before balancing
        n_trials      : dict (stim_idx, cond) -> int   count actually used
        n_balanced    : int or None  common count per cell (None if balance=False)
        cond_values   : sorted list of unique condition values found
        run_dir       : Path
    or None if no data found.
    """
    pkl_path = run_dir / 'metrics_numpy.pkl'
    if not pkl_path.exists():
        print(f'  WARNING: {pkl_path} not found')
        return None
    with open(pkl_path, 'rb') as f:
        m = pickle.load(f)

    accum = defaultdict(list)   # (stim_idx, cond_val) -> list of (H,) vecs in trial order

    for tb in m['trial_boundaries']:
        ti       = int(tb['trial_idx'])
        stim_idx = int(tb['stimulus'])
        phase_i  = int(tb['phase_idx'])
        rev_ph   = int(tb['reversal_phase'])

        # ── filters ──────────────────────────────────────────────────────────
        if select_phases is not None and phase_i not in select_phases:
            continue
        if trial_min is not None and ti < trial_min:
            continue
        if trial_max is not None and ti > trial_max:
            continue
        if stim_idx not in plot_stims:
            continue

        hs_list = m['hidden_states'].get(ti)
        ws_list = m['within_trial_states'].get(ti)
        if not hs_list or not ws_list:
            continue

        hs     = np.array(hs_list)            # (T, H)
        states = list(ws_list)
        stim_m = np.array([s in _STIM_STATES for s in states])
        if stim_m.sum() == 0:
            continue

        cond_val = rev_ph if cond_by == 'reversal_phase' else phase_i
        accum[(stim_idx, cond_val)].append(hs[stim_m].mean(0))   # (H,)

    if not accum:
        return None

    n_trials_raw = {k: len(v) for k, v in accum.items()}

    # ── optional balancing: truncate every cell to the minimum count ──────────
    if balance:
        n_balanced = min(n_trials_raw.values())
        accum = {k: v[:n_balanced] for k, v in accum.items()}
    else:
        n_balanced = None

    cond_means = {k: np.mean(v, axis=0) for k, v in accum.items()}
    n_trials   = {k: len(v)             for k, v in accum.items()}
    cond_values = sorted(set(c for _, c in accum.keys()))
    return {'cond_means':   cond_means,
            'n_trials_raw': n_trials_raw,
            'n_trials':     n_trials,
            'n_balanced':   n_balanced,
            'cond_values':  cond_values,
            'run_dir':      run_dir}


def selectivity_index(T, eps=1e-8):
    """SI per unit given tuning matrix T of shape (n_stims, H)."""
    return (T.max(0) - T.min(0)) / (np.abs(T.max(0)) + np.abs(T.min(0)) + eps)


def build_tuning_matrices(cond_means, plot_stims, cond_values):
    """
    Build per-condition and mean tuning matrices.

    Returns
    -------
    T_per_cond : dict  cond_val -> (n_stims, H)
    T_mean     : (n_stims, H)  mean across conditions
    """
    H = next(iter(cond_means.values())).shape[0]
    T_per_cond = {}
    for cv in cond_values:
        rows = []
        for s in plot_stims:
            v = cond_means.get((s, cv))
            rows.append(v if v is not None else np.zeros(H))
        T_per_cond[cv] = np.stack(rows)   # (n_stims, H)
    T_mean = np.stack(list(T_per_cond.values())).mean(0)
    return T_per_cond, T_mean

In [ ]:
BALANCE_TRIALS = True   # truncate all (stim, cond) cells to the minimum count

run_data = []
for i, rd in enumerate(run_dirs):
    r = compute_run_tuning(rd, plot_stims=PLOT_STIMS,
                           select_phases=SELECT_PHASES,
                           trial_min=TRIAL_MIN, trial_max=TRIAL_MAX,
                           cond_by=COND_BY, balance=BALANCE_TRIALS)
    if r is None:
        print(f'Run {i:2d}: WARNING — no data, skipping')
        continue
    r['run_i'] = i

    T_per_cond, T_mean = build_tuning_matrices(
        r['cond_means'], PLOT_STIMS, r['cond_values'])
    r['T_per_cond']  = T_per_cond
    r['T_mean']      = T_mean
    r['si_mean']     = selectivity_index(T_mean)
    r['si_per_cond'] = {cv: selectivity_index(T) for cv, T in T_per_cond.items()}
    r['silent_mask'] = T_mean.max(0) < SILENT_THR
    run_data.append(r)

print(f'{len(run_data)} / {len(run_dirs)} runs loaded OK')
if not run_data:
    raise RuntimeError('No run data — check SELECT_PHASES / TRIAL_MIN / TRIAL_MAX.')

In [ ]:
# ── Trial count table — raw counts and balanced count per run ─────────────────
# Columns: one per (stim, cond) cell.  Rows: one per run.
# Green cell = minimum (bottleneck); orange = within 10% of min; red = >10% above min.

cond_vals   = run_data[0]['cond_values']
cond_lbl    = {cv: (f'cond{cv}' if COND_BY == 'phase_idx' else ('pre' if cv == 0 else 'post'))
               for cv in cond_vals}
stim_lbl    = {s: STIM_LETTER[s] for s in PLOT_STIMS}
cell_keys   = [(s, cv) for s in PLOT_STIMS for cv in cond_vals]
col_headers = [f'{stim_lbl[s]}/{cond_lbl[cv]}' for s, cv in cell_keys]

# Header
col_w = 12
run_w = 8
hdr   = f"{'Run':<{run_w}}" + ''.join(f'{h:>{col_w}}' for h in col_headers)
hdr  += f"{'min':>{col_w}}{'used':>{col_w}}"
if BALANCE_TRIALS:
    hdr += f"  (balanced → {run_data[0]['n_balanced']} per cell)"
print(hdr)
print('─' * len(hdr.split('(')[0]))

all_mins = []
for r in run_data:
    raw   = r['n_trials_raw']
    used  = r['n_trials']
    rmin  = min(raw.values())
    all_mins.append(rmin)
    row   = f"{r['run_dir'].name:<{run_w}}"
    for key in cell_keys:
        n_raw = raw.get(key, 0)
        flag  = ''
        if n_raw == rmin:
            flag = '*'   # bottleneck cell
        elif n_raw > rmin * 1.10:
            flag = '+'   # >10% above min (would be dropped if balancing)
        row += f"{str(n_raw) + flag:>{col_w}}"
    row += f"{rmin:>{col_w}}{used.get(cell_keys[0], 0):>{col_w}}"
    print(row)

print('─' * len(hdr.split('(')[0]))
print(f"{'':>{run_w}}" +
      ''.join(f"{np.mean([r['n_trials_raw'].get(k,0) for r in run_data]):>{col_w}.0f}"
              for k in cell_keys) +
      f"  ← mean raw per cell")
print()
print(f'* = bottleneck cell (minimum count, determines balanced n)')
print(f'+ = >10% above minimum (trials dropped when BALANCE_TRIALS=True)')
if BALANCE_TRIALS:
    n_bal = run_data[0]['n_balanced']
    max_raw = max(max(r['n_trials_raw'].values()) for r in run_data)
    pct_dropped = 100 * (1 - n_bal / max_raw)
    print(f'\nBalanced to {n_bal} trials per cell  '
          f'({pct_dropped:.1f}% of largest cell dropped at most)')

---
## 3. SI threshold config

Set `SI_THRESHOLD` (absolute) **or** `SI_PERCENTILE` (top-N% of active units).  
One must be `None`.

In [ ]:
SI_THRESHOLD  = 0.25   # absolute threshold  (set to None to use percentile)
SI_PERCENTILE = None   # e.g. 75 → top 25% of active units


def get_selective_mask(si_vec, silent_mask):
    active = ~silent_mask
    if SI_THRESHOLD is not None:
        return active & (si_vec >= SI_THRESHOLD)
    cutoff = np.percentile(si_vec[active], SI_PERCENTILE)
    return active & (si_vec >= cutoff)


for r in run_data:
    r['sel_mask']  = {}
    r['sel_props'] = {}
    for label, T, si in (
            [('mean', r['T_mean'], r['si_mean'])]
            + [(cv, r['T_per_cond'][cv], r['si_per_cond'][cv])
               for cv in r['cond_values']]):
        mask  = get_selective_mask(si, r['silent_mask'])
        pref  = T.argmax(0)
        n_sel = int(mask.sum())
        props = np.array([np.sum(mask & (pref == j)) / max(n_sel, 1)
                          for j in range(len(PLOT_STIMS))])
        r['sel_mask'][label]  = mask
        r['sel_props'][label] = props

thresh_str = (f'SI ≥ {SI_THRESHOLD}' if SI_THRESHOLD is not None
              else f'top {100 - SI_PERCENTILE:.0f}th percentile')
print(f'Threshold: {thresh_str}')
for r in run_data:
    n_act = int((~r['silent_mask']).sum())
    n_sel = int(r['sel_mask']['mean'].sum())
    print(f'  Run {r["run_i"]}: active={n_act}  selective={n_sel} ({100*n_sel/max(n_act,1):.0f}%)')

---
## 4. Tuning heatmaps — per run

One panel per condition (context).  
Active units are sorted by preferred stimulus (highest mean response), then by SI
(descending). Silent units appended at right in grey.  
Activation is z-scored across units (within each condition panel).

In [ ]:
stim_names_ordered = [STIM_NAMES[s] for s in PLOT_STIMS]
cmap_heat = plt.cm.RdBu_r.copy()

for r in run_data:
    active   = ~r['silent_mask']
    n_active = int(active.sum())
    n_silent = int(r['silent_mask'].sum())
    H        = len(r['si_mean'])

    # Sort active units only: by preferred stimulus, then SI descending
    pref_active = r['T_mean'].argmax(0)[active]
    si_active   = r['si_mean'][active]
    sort_active = np.lexsort((-si_active, pref_active))
    active_idx  = np.where(active)[0][sort_active]
    # Silent units excluded from heatmap entirely — just noted in the title

    n_conds = len(r['cond_values'])
    fig, axes = plt.subplots(1, n_conds, figsize=(6 * n_conds + 1, 4.5), sharey=True)
    if n_conds == 1:
        axes = [axes]

    for ax, cv in zip(axes, r['cond_values']):
        T_sorted = r['T_per_cond'][cv][:, active_idx]   # (n_stims, n_active) — active only
        mu  = T_sorted.mean(0, keepdims=True)
        sig = T_sorted.std(0,  keepdims=True) + 1e-8
        T_z = (T_sorted - mu) / sig

        im = ax.imshow(T_z, aspect='auto', cmap=cmap_heat,
                       vmin=-2, vmax=2, interpolation='nearest')
        ax.set_yticks(range(len(stim_names_ordered)))
        ax.set_yticklabels(stim_names_ordered)
        ax.set_xlabel('Active hidden unit (sorted: pref. stim → SI desc.)')

        cond_lbl = (CTX_TITLES.get(cv, f'cond {cv}') if COND_BY == 'reversal_phase'
                    else f'Phase {cv}')
        n_t = sum(r['n_trials'].get((s, cv), 0) for s in PLOT_STIMS)
        ax.set_title(f'{cond_lbl}\n({n_t} trials)', fontsize=9)

        # Dashed lines at stim-group boundaries
        pref_sorted = pref_active[sort_active]
        for b in np.where(np.diff(pref_sorted))[0] + 0.5:
            ax.axvline(b, color='white', lw=1.5, ls=':', zorder=3)

        # Lime line at SI threshold boundary
        n_sel_active = int((r['sel_mask'].get(cv, np.zeros(H, bool)))[active_idx].sum())
        if n_sel_active > 0:
            ax.axvline(n_sel_active - 0.5, color='lime', lw=1.5, ls='--', zorder=4,
                       label=f'{thresh_str} → {n_sel_active} units')
            ax.legend(fontsize=7, loc='upper right')

        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03).set_label('z-score')

    axes[0].set_ylabel('Stimulus')
    silent_note = f'  |  silent (excluded): {n_silent}' if n_silent > 0 else ''
    fig.suptitle(
        f'Run {r["run_i"]} — tuning heatmap  (active: {n_active}{silent_note})\n'
        f'Variant: {VARIANT_DIR.name}  |  '
        f'phases={SELECT_PHASES}  |  trials=[{TRIAL_MIN},{TRIAL_MAX}]',
        fontsize=9)
    plt.tight_layout()
    save_fig(fig, f'4_tuning_heatmap_run{r["run_i"]:02d}.png')
    plt.show()

---
## 5. Average tuning heatmap across runs

Units are aligned across runs by **preferred stimulus then SI rank** within each run,
then all runs are stacked and averaged (after z-scoring within each run).  
Because different runs may have different preferred-stim compositions the per-run
normalised heatmaps are averaged in sorted order.

In [ ]:
# Build per-run z-scored, sorted, mean-condition tuning matrices
all_T_z_sorted = []   # list of (n_stims, H) z-scored matrices per run

for r in run_data:
    active  = ~r['silent_mask']
    n_active = int(active.sum())
    pref_a  = r['T_mean'].argmax(0)[active]
    si_a    = r['si_mean'][active]
    sort_a  = np.lexsort((-si_a, pref_a))
    ai      = np.where(active)[0][sort_a]

    T = r['T_mean'][:, ai]   # (n_stims, n_active)
    mu  = T.mean(0, keepdims=True)
    sig = T.std(0, keepdims=True) + 1e-8
    all_T_z_sorted.append((T - mu) / sig)

# Pad each run's matrix to the same width (max n_active across runs)
H_max = max(T.shape[1] for T in all_T_z_sorted)
padded = []
for T in all_T_z_sorted:
    pad_w = H_max - T.shape[1]
    padded.append(np.pad(T, ((0,0),(0,pad_w)), constant_values=np.nan))

mean_T_z = np.nanmean(np.stack(padded), axis=0)  # (n_stims, H_max)
cond_labels_avg = [STIM_NAMES[s] for s in PLOT_STIMS]

fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))
im = ax.imshow(mean_T_z, aspect='auto', cmap=cmap_heat,
               vmin=-1.5, vmax=1.5, interpolation='nearest')
ax.set_yticks(range(len(cond_labels_avg)))
ax.set_yticklabels(cond_labels_avg)
ax.set_xlabel('Hidden unit rank (within-run sort: pref. stim → SI desc.)')
ax.set_ylabel('Stimulus')
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02).set_label('Mean z-score')

fig.suptitle(
    f'Average tuning heatmap — {len(run_data)} runs  |  {VARIANT_DIR.name}\n'
    f'Mean across runs (each z-scored, units sorted by pref-stim then SI within run)',
    fontsize=10)
plt.tight_layout()
save_fig(fig, '5_avg_tuning_heatmap.png')
plt.show()

---
## 6. SI distributions

Per run and combined. Active units only (silent units excluded from all distributions).

In [ ]:
n_r = len(run_data)
ncols = min(n_r, 5)
nrows = int(np.ceil(n_r / ncols))
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(4.5 * ncols, 3.5 * nrows), squeeze=False)

bins = np.linspace(0, 1, 31)
cond_hist_colors = {cv: CTX_COLORS.get(cv, f'C{cv}') for cv in run_data[0]['cond_values']}
cond_hist_labels = {cv: (CTX_TITLES.get(cv, f'Cond {cv}') if COND_BY == 'reversal_phase'
                         else f'Phase {cv}')
                    for cv in run_data[0]['cond_values']}

for flat_i, r in enumerate(run_data):
    row, col = divmod(flat_i, ncols)
    ax  = axes[row, col]
    act = ~r['silent_mask']

    ax.hist(r['si_mean'][act], bins=bins, color='#555', alpha=0.5,
            label='Mean (all conds)', density=True)
    for cv in r['cond_values']:
        ax.hist(r['si_per_cond'][cv][act], bins=bins,
                color=cond_hist_colors[cv], alpha=0.4,
                label=cond_hist_labels[cv], density=True)

    med = np.median(r['si_mean'][act]) if act.sum() > 0 else float('nan')
    ax.axvline(med, color='k', lw=1.5, ls='--', label=f'Median={med:.2f}')
    if SI_THRESHOLD is not None:
        ax.axvline(SI_THRESHOLD, color='lime', lw=1.5, ls=':', label=f'Threshold={SI_THRESHOLD}')

    n_sil = int(r['silent_mask'].sum())
    n_tot = len(r['si_mean'])
    ax.text(0.97, 0.97, f'silent: {n_sil}/{n_tot}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7, color='#666',
            bbox=dict(boxstyle='round,pad=0.2', fc='#eee', ec='none'))
    ax.set_xlim(0, 1)
    ax.set_xlabel('SI')
    ax.set_ylabel('Density' if col == 0 else '')
    ax.set_title(f'Run {r["run_i"]}', fontsize=9)
    if flat_i == 0:
        ax.legend(fontsize=6)

for flat_i in range(len(run_data), nrows * ncols):
    row, col = divmod(flat_i, ncols)
    axes[row, col].set_visible(False)

fig.suptitle(f'SI distributions — {VARIANT_DIR.name}\n'
             f'Active units only  |  phases={SELECT_PHASES}  '
             f'trials=[{TRIAL_MIN},{TRIAL_MAX}]', fontsize=10)
plt.tight_layout()
save_fig(fig, '6_si_distributions_per_run.png')
plt.show()

In [ ]:
# Combined SI distribution across all runs (pooled active units)
all_si_pooled = np.concatenate([r['si_mean'][~r['silent_mask']] for r in run_data])

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(all_si_pooled, bins=np.linspace(0, 1, 41), color='#555', alpha=0.7,
        density=True, label=f'All runs pooled (n={len(all_si_pooled)} units)')

# Per-run median lines (light)
for r in run_data:
    act = ~r['silent_mask']
    if act.sum() > 0:
        ax.axvline(np.median(r['si_mean'][act]), color='steelblue', lw=0.8, alpha=0.4)

grand_med = np.median(all_si_pooled)
ax.axvline(grand_med, color='navy', lw=2, ls='--', label=f'Grand median = {grand_med:.3f}')
if SI_THRESHOLD is not None:
    ax.axvline(SI_THRESHOLD, color='lime', lw=2, ls=':', label=f'Threshold = {SI_THRESHOLD}')
    pct_above = 100 * np.mean(all_si_pooled >= SI_THRESHOLD)
    ax.text(SI_THRESHOLD + 0.01, ax.get_ylim()[1] * 0.9,
            f'{pct_above:.0f}% selective', fontsize=9, color='green')

ax.set_xlabel('Selectivity index (active units)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
fig.suptitle(f'Combined SI distribution — {len(run_data)} runs\n'
             f'{VARIANT_DIR.name}  |  phases={SELECT_PHASES}',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '6b_si_distribution_combined.png')
plt.show()

---
## 7. Selective-unit proportions

For each stimulus: fraction of selective units (SI ≥ threshold) whose preferred stimulus
is that stimulus.  Shown per run and as mean ± SEM.

In [ ]:
bar_labels  = [STIM_NAMES[s] for s in PLOT_STIMS]
bar_colors  = [STIM_COLORS[s] for s in PLOT_STIMS]
x_bars      = np.arange(len(PLOT_STIMS))
chance      = 1.0 / len(PLOT_STIMS)

cond_keys = list(run_data[0]['cond_values']) + ['mean']
cond_labels_plot = {cv: (CTX_TITLES.get(cv, f'Cond {cv}') if COND_BY == 'reversal_phase'
                         else f'Phase {cv}')
                    for cv in run_data[0]['cond_values']}
cond_labels_plot['mean'] = 'Mean (all conds)'

n_panels = len(cond_keys)
n_r = len(run_data)
fig, axes = plt.subplots(n_r, n_panels,
                         figsize=(4.5 * n_panels, 3.5 * n_r), squeeze=False)

for row, r in enumerate(run_data):
    n_act = int((~r['silent_mask']).sum())
    for col, key in enumerate(cond_keys):
        ax    = axes[row, col]
        props = r['sel_props'][key]
        n_sel = int(r['sel_mask'][key].sum())
        bars  = ax.bar(x_bars, props, color=bar_colors,
                       edgecolor='k', linewidth=0.7, width=0.6)
        ax.axhline(chance, color='grey', lw=1.2, ls='--', label=f'Chance ({chance:.2f})')
        for bar, p in zip(bars, props):
            nu = int(round(p * n_sel))
            if nu > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.01, str(nu),
                        ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x_bars)
        ax.set_xticklabels(bar_labels, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Proportion' if col == 0 else '')
        ax.set_ylim(0, min(1.0, props.max() * 1.5 + 0.05))
        ax.set_title(f'Run {r["run_i"]} | {cond_labels_plot[key]}\n'
                     f'sel={n_sel}/{n_act} ({100*n_sel/max(n_act,1):.0f}%)',
                     fontsize=8)
        if row == 0 and col == 0:
            ax.legend(fontsize=7)

fig.suptitle(f'Selective-unit proportions per run — {thresh_str}\n'
             f'{VARIANT_DIR.name}', fontsize=10, y=1.01)
plt.tight_layout()
save_fig(fig, '7a_sel_proportions_per_run.png')
plt.show()

In [ ]:
# Mean ± SEM across runs
props_stack = {key: np.stack([r['sel_props'][key] for r in run_data])
               for key in cond_keys}
n_sel_stack  = {key: np.array([r['sel_mask'][key].sum() for r in run_data])
                for key in cond_keys}
n_act_arr    = np.array([(~r['silent_mask']).sum() for r in run_data])
n_r = len(run_data)

fig, axes = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 4.5), squeeze=False)

for col, key in enumerate(cond_keys):
    ax     = axes[0, col]
    P      = props_stack[key]
    mean_p = P.mean(0)
    sem_p  = P.std(0) / np.sqrt(n_r)

    bars = ax.bar(x_bars, mean_p, yerr=sem_p,
                  color=bar_colors, edgecolor='k', linewidth=0.7, width=0.6,
                  error_kw=dict(ecolor='k', capsize=4, lw=1.2))
    ax.axhline(chance, color='grey', lw=1.2, ls='--', label=f'Chance ({chance:.2f})')

    mean_sel = n_sel_stack[key].mean()
    for bar, mp, sp in zip(bars, mean_p, sem_p):
        nu = int(round(mp * mean_sel))
        if nu > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + sp + 0.01, str(nu),
                    ha='center', va='bottom', fontsize=8)

    # Individual run dots
    for r in run_data:
        ax.plot(x_bars, r['sel_props'][key], 'o',
                color='k', markersize=3, alpha=0.35, zorder=5)

    mean_act = n_act_arr.mean()
    ax.set_title(f'{cond_labels_plot[key]}\n'
                 f'mean sel = {mean_sel:.0f} / {mean_act:.0f} '
                 f'({100*mean_sel/max(mean_act,1):.0f}%)', fontsize=9)
    ax.set_xticks(x_bars)
    ax.set_xticklabels(bar_labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Proportion of selective units' if col == 0 else '')
    ax.set_ylim(0, min(1.0, (mean_p + sem_p).max() * 1.5 + 0.05))
    if col == 0:
        ax.legend(fontsize=8)

fig.suptitle(f'Selective-unit proportions — mean ± SEM  ({n_r} runs)\n'
             f'{VARIANT_DIR.name}  |  {thresh_str}  |  dots = individual runs',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '7b_sel_proportions_mean_sem.png')
plt.show()

---
## 8. SI threshold sweep

For each threshold value in a fine grid, compute:
- Total fraction of active units classified as selective (all runs, mean ± SEM).
- Fraction of selective units preferring each stimulus (mean ± SEM across runs).

This shows how selectivity composition changes as the criterion becomes stricter.

In [ ]:
thr_grid = np.linspace(0.0, 0.95, 100)

# Shape: (n_thresholds, n_runs, n_stims)
frac_selective  = np.zeros((len(thr_grid), len(run_data)))
stim_fracs      = np.zeros((len(thr_grid), len(run_data), len(PLOT_STIMS)))

for ti, thr in enumerate(thr_grid):
    for ri, r in enumerate(run_data):
        act   = ~r['silent_mask']
        n_act = int(act.sum())
        si    = r['si_mean']
        mask  = act & (si >= thr)
        n_sel = int(mask.sum())
        frac_selective[ti, ri] = n_sel / max(n_act, 1)
        pref = r['T_mean'].argmax(0)
        for ji in range(len(PLOT_STIMS)):
            stim_fracs[ti, ri, ji] = np.sum(mask & (pref == ji)) / max(n_sel, 1)

mean_frac_sel = frac_selective.mean(1)
sem_frac_sel  = frac_selective.std(1)  / np.sqrt(len(run_data))
mean_sf       = stim_fracs.mean(1)     # (n_thr, n_stims)
sem_sf        = stim_fracs.std(1)      / np.sqrt(len(run_data))

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)

# ── Top panel: fraction of active units selective ─────────────────────────────
ax = axes[0]
ax.fill_between(thr_grid,
                mean_frac_sel - sem_frac_sel,
                mean_frac_sel + sem_frac_sel,
                color='#666', alpha=0.25)
ax.plot(thr_grid, mean_frac_sel, color='#333', lw=2,
        label=f'Mean ± SEM  (n={len(run_data)} runs)')
# Individual run traces
for ri in range(len(run_data)):
    ax.plot(thr_grid, frac_selective[:, ri], lw=0.6, alpha=0.35, color='steelblue')

if SI_THRESHOLD is not None:
    ax.axvline(SI_THRESHOLD, color='lime', lw=2, ls='--', label=f'Current ({SI_THRESHOLD})')

ax.set_ylabel('Fraction of active units classified selective')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.set_title('Proportion selective vs SI threshold')

# ── Bottom panel: proportion of selective units preferring each stim ──────────
ax = axes[1]
for ji, (stim_idx, lbl, col) in enumerate(zip(PLOT_STIMS, bar_labels, bar_colors)):
    ax.fill_between(thr_grid,
                    mean_sf[:, ji] - sem_sf[:, ji],
                    mean_sf[:, ji] + sem_sf[:, ji],
                    color=col, alpha=0.2)
    ax.plot(thr_grid, mean_sf[:, ji], color=col, lw=2, label=lbl)

ax.axhline(chance, color='grey', lw=1.2, ls=':', label=f'Chance ({chance:.2f})')
if SI_THRESHOLD is not None:
    ax.axvline(SI_THRESHOLD, color='lime', lw=2, ls='--', label=f'Current ({SI_THRESHOLD})')

ax.set_xlabel('SI threshold')
ax.set_ylabel('Fraction of selective units')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.set_title('Preferred-stimulus composition vs SI threshold')

fig.suptitle(f'SI threshold sweep — mean ± SEM  ({len(run_data)} runs)\n'
             f'{VARIANT_DIR.name}  |  phases={SELECT_PHASES}',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '8_si_threshold_sweep.png')
plt.show()

---
## 9. Compare across all variants (hidden sizes)

For each variant in the sweep, compute SI statistics and selective-unit proportions
at the chosen threshold.  Useful for seeing how model capacity affects tuning.

In [ ]:
# Extract hidden size from variant name (looks for _h<N>_)
import re

def _get_hidden_size(vdir):
    m = re.search(r'_h(\d+)_', vdir.name)
    return int(m.group(1)) if m else -1

sweep_results = []
for vdir in variant_dirs:
    h = _get_hidden_size(vdir)
    rdirs = sorted([d for d in vdir.iterdir()
                    if d.is_dir() and d.name.startswith('run_')])
    if not rdirs:
        continue
    run_data_v = []
    for rd in rdirs:
        r = compute_run_tuning(rd, plot_stims=PLOT_STIMS,
                               select_phases=SELECT_PHASES,
                               trial_min=TRIAL_MIN, trial_max=TRIAL_MAX,
                               cond_by=COND_BY)
        if r is None:
            continue
        T_per_cond, T_mean = build_tuning_matrices(
            r['cond_means'], PLOT_STIMS, r['cond_values'])
        r['T_per_cond']  = T_per_cond
        r['T_mean']      = T_mean
        r['si_mean']     = selectivity_index(T_mean)
        r['si_per_cond'] = {cv: selectivity_index(T) for cv, T in T_per_cond.items()}
        r['silent_mask'] = T_mean.max(0) < SILENT_THR
        run_data_v.append(r)
    if run_data_v:
        sweep_results.append({'hidden_size': h, 'vdir': vdir, 'run_data': run_data_v})
        print(f'h={h:4d}  {len(run_data_v)} runs  {vdir.name}')

sweep_results.sort(key=lambda x: x['hidden_size'])
print(f'\nLoaded {len(sweep_results)} variants')

In [ ]:
hidden_sizes = [s['hidden_size'] for s in sweep_results]

# Per-variant statistics
median_si_mean_v = []
frac_sel_mean_v  = []
frac_sel_sem_v   = []
stim_prop_mean_v = []   # (n_variants, n_stims)
stim_prop_sem_v  = []

for s in sweep_results:
    rd = s['run_data']
    # Median SI across all active units pooled over runs
    all_si = np.concatenate([r['si_mean'][~r['silent_mask']] for r in rd])
    median_si_mean_v.append(np.median(all_si))

    # Fraction selective per run, then mean ± SEM
    fracs = []
    sprops = []
    for r in rd:
        act   = ~r['silent_mask']
        n_act = int(act.sum())
        mask  = act & (r['si_mean'] >= (SI_THRESHOLD or 0))
        n_sel = int(mask.sum())
        fracs.append(n_sel / max(n_act, 1))
        pref = r['T_mean'].argmax(0)
        sprops.append([np.sum(mask & (pref == ji)) / max(n_sel, 1)
                       for ji in range(len(PLOT_STIMS))])
    frac_sel_mean_v.append(np.mean(fracs))
    frac_sel_sem_v.append(np.std(fracs) / np.sqrt(len(fracs)))
    stim_prop_mean_v.append(np.mean(sprops, axis=0))
    stim_prop_sem_v.append(np.std(sprops, axis=0) / np.sqrt(len(sprops)))

stim_prop_mean_v = np.array(stim_prop_mean_v)  # (n_var, n_stims)
stim_prop_sem_v  = np.array(stim_prop_sem_v)
x_v = np.arange(len(hidden_sizes))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: median SI vs hidden size
axes[0].plot(x_v, median_si_mean_v, 'o-', color='#333', lw=2, ms=7)
axes[0].set_xticks(x_v); axes[0].set_xticklabels(hidden_sizes)
axes[0].set_xlabel('Hidden size'); axes[0].set_ylabel('Median SI (active units pooled)')
axes[0].set_title('Selectivity vs model size')

# Panel 2: fraction selective vs hidden size
axes[1].errorbar(x_v, frac_sel_mean_v, yerr=frac_sel_sem_v,
                 fmt='o-', color='#333', lw=2, capsize=4, ms=7)
axes[1].set_xticks(x_v); axes[1].set_xticklabels(hidden_sizes)
axes[1].set_xlabel('Hidden size')
axes[1].set_ylabel(f'Frac. selective ({thresh_str})')
axes[1].set_title('Fraction selective vs model size')
axes[1].set_ylim(0, 1.05)

# Panel 3: preferred-stim composition vs hidden size
for ji, (stim_idx, lbl, col) in enumerate(zip(PLOT_STIMS, bar_labels, bar_colors)):
    axes[2].fill_between(x_v,
                         stim_prop_mean_v[:, ji] - stim_prop_sem_v[:, ji],
                         stim_prop_mean_v[:, ji] + stim_prop_sem_v[:, ji],
                         color=col, alpha=0.2)
    axes[2].plot(x_v, stim_prop_mean_v[:, ji], 'o-', color=col, lw=2, ms=7,
                 label=lbl)
axes[2].axhline(chance, color='grey', lw=1.2, ls=':', label=f'Chance ({chance:.2f})')
axes[2].set_xticks(x_v); axes[2].set_xticklabels(hidden_sizes)
axes[2].set_xlabel('Hidden size')
axes[2].set_ylabel('Proportion of selective units')
axes[2].set_title('Pref-stim composition vs model size')
axes[2].legend(fontsize=8)
axes[2].set_ylim(0, 1.05)

fig.suptitle(f'Model-sweep comparison — {thresh_str}  |  phases={SELECT_PHASES}', fontsize=11)
plt.tight_layout()
save_fig(fig, '9_model_sweep_comparison.png')
plt.show()